In [1]:
# ============================================================
# WEEK 8/FINAL — HUMAN EVALUATION
# Cell 1: Imports and project paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\Admin\Capstone_Project"
)

# ------------------------------------------------------------
# Human-labeled input data
# ------------------------------------------------------------

HUMAN_LABEL_DIR = (
    PROJECT_ROOT
    / "Outputs"
    / "Tables"
    / "Human_Labelling"
)

# ------------------------------------------------------------
# Final human-evaluation outputs
# ------------------------------------------------------------

FINAL_EVAL_DIR = (
    PROJECT_ROOT
    / "Outputs"
    / "Tables"
    / "Human_Evaluation"
)

FINAL_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Display paths
# ------------------------------------------------------------

print("=" * 70)
print("FINAL HUMAN EVALUATION — PROJECT PATHS")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nHuman-labeled data directory:")
print(HUMAN_LABEL_DIR)

print("\nHuman-evaluation output directory:")
print(FINAL_EVAL_DIR)

print("\nHuman-labeled data directory exists:")
print(HUMAN_LABEL_DIR.exists())

print("\nFinal evaluation directory exists:")
print(FINAL_EVAL_DIR.exists())

FINAL HUMAN EVALUATION — PROJECT PATHS

Project root:
C:\Users\Admin\Capstone_Project

Human-labeled data directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Human_Labelling

Human-evaluation output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Human_Evaluation

Human-labeled data directory exists:
True

Final evaluation directory exists:
True


In [3]:
# ============================================================
# Cell 2: Load completed human-label datasets
# ============================================================

rag_file = HUMAN_LABEL_DIR / "week7_rag_human_review_labeled.csv"

ranking_file = HUMAN_LABEL_DIR / "week7_recommendation_ranking_ground_truth_labeled_filled.csv"

skill_extraction_file = HUMAN_LABEL_DIR / "week7_skill_extraction_ground_truth_labeled_REVIEW_filled.xlsx"

skill_gap_file = HUMAN_LABEL_DIR / "week7_skill_gap_ground_truth_labeled.csv"


# Load datasets
rag_df = pd.read_csv(rag_file)

ranking_df = pd.read_csv(ranking_file)

skill_extraction_df = pd.read_excel(skill_extraction_file)

skill_gap_df = pd.read_csv(skill_gap_file)


# Display shapes
print("RAG shape:", rag_df.shape)
print("Ranking shape:", ranking_df.shape)
print("Skill Extraction shape:", skill_extraction_df.shape)
print("Skill Gap shape:", skill_gap_df.shape)

RAG shape: (20, 23)
Ranking shape: (300, 13)
Skill Extraction shape: (200, 8)
Skill Gap shape: (200, 17)


In [5]:
# ============================================================
# Cell 3: Validate human-label completeness
# ============================================================

# Skill Extraction
skill_extraction_complete = skill_extraction_df["human_skill_present"].notna().sum()

# Recommendation Ranking
ranking_relevant_complete = ranking_df["human_relevant"].notna().sum()
ranking_grade_complete = ranking_df["human_relevance_grade"].notna().sum()

# Skill Gap
skill_gap_complete = skill_gap_df["human_gap"].notna().sum()
skill_gap_severity_complete = skill_gap_df["human_gap_severity"].notna().sum()
skill_gap_expert_complete = skill_gap_df["expert_rating"].notna().sum()

# RAG
rag_rating_columns = [
    "relevance_rating",
    "correctness_rating",
    "groundedness_rating",
    "citation_correctness_rating",
    "completeness_rating",
    "clarity_rating",
    "usefulness_rating"
]

print("Skill Extraction labels:",
      f"{skill_extraction_complete}/{len(skill_extraction_df)}")

print("Ranking relevance labels:",
      f"{ranking_relevant_complete}/{len(ranking_df)}")

print("Ranking grade labels:",
      f"{ranking_grade_complete}/{len(ranking_df)}")

print("Skill Gap labels:",
      f"{skill_gap_complete}/{len(skill_gap_df)}")

print("Skill Gap severity labels:",
      f"{skill_gap_severity_complete}/{len(skill_gap_df)}")

print("Skill Gap expert ratings:",
      f"{skill_gap_expert_complete}/{len(skill_gap_df)}")

for col in rag_rating_columns:
    print(f"RAG {col}:",
          f"{rag_df[col].notna().sum()}/{len(rag_df)}")

print("RAG unsupported claim count:",
      f"{rag_df['unsupported_claim_count'].notna().sum()}/{len(rag_df)}")

Skill Extraction labels: 200/200
Ranking relevance labels: 300/300
Ranking grade labels: 300/300
Skill Gap labels: 200/200
Skill Gap severity labels: 200/200
Skill Gap expert ratings: 200/200
RAG relevance_rating: 20/20
RAG correctness_rating: 20/20
RAG groundedness_rating: 20/20
RAG citation_correctness_rating: 20/20
RAG completeness_rating: 20/20
RAG clarity_rating: 20/20
RAG usefulness_rating: 20/20
RAG unsupported claim count: 20/20


In [7]:
# ============================================================
# Cell 4: Skill Extraction Evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Ground truth and system prediction
y_true = skill_extraction_df["human_skill_present"].astype(int)
y_pred = skill_extraction_df["system_extracted"].astype(int)

# Metrics
skill_accuracy = accuracy_score(y_true, y_pred)
skill_precision = precision_score(y_true, y_pred, zero_division=0)
skill_recall = recall_score(y_true, y_pred, zero_division=0)
skill_f1 = f1_score(y_true, y_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("SKILL EXTRACTION EVALUATION")
print("-" * 40)
print(f"Accuracy : {skill_accuracy:.4f}")
print(f"Precision: {skill_precision:.4f}")
print(f"Recall   : {skill_recall:.4f}")
print(f"F1 Score : {skill_f1:.4f}")

print("\nConfusion Matrix Counts")
print(f"TP: {tp}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TN: {tn}")

SKILL EXTRACTION EVALUATION
----------------------------------------
Accuracy : 0.8350
Precision: 0.8811
Recall   : 0.8873
F1 Score : 0.8842

Confusion Matrix Counts
TP: 126
FP: 17
FN: 16
TN: 41


In [9]:
# ============================================================
# Cell 5: Recommendation Ranking Evaluation
# ============================================================

def evaluate_candidate_ranking(group):
    group = group.sort_values("system_rank").copy()

    # Binary human relevance
    relevance = group["human_relevant"].astype(int).to_numpy()

    # Graded human relevance (0–3)
    grades = group["human_relevance_grade"].astype(float).to_numpy()

    k = min(5, len(group))

    # -----------------------------
    # Precision@5
    # -----------------------------
    precision_at_5 = relevance[:k].sum() / k

    # -----------------------------
    # Recall@5
    # -----------------------------
    total_relevant = relevance.sum()

    recall_at_5 = (
        relevance[:k].sum() / total_relevant
        if total_relevant > 0
        else np.nan
    )

    # -----------------------------
    # NDCG@5
    # -----------------------------
    discounts = 1 / np.log2(np.arange(2, k + 2))

    dcg = np.sum(
        ((2 ** grades[:k]) - 1) * discounts
    )

    ideal_grades = np.sort(grades)[::-1][:k]

    idcg = np.sum(
        ((2 ** ideal_grades) - 1) * discounts
    )

    ndcg_at_5 = dcg / idcg if idcg > 0 else np.nan

    # -----------------------------
    # Reciprocal Rank
    # -----------------------------
    relevant_positions = np.where(relevance == 1)[0]

    reciprocal_rank = (
        1 / (relevant_positions[0] + 1)
        if len(relevant_positions) > 0
        else 0
    )

    return {
        "precision_at_5": precision_at_5,
        "recall_at_5": recall_at_5,
        "ndcg_at_5": ndcg_at_5,
        "reciprocal_rank": reciprocal_rank
    }


# ============================================================
# Evaluate each candidate separately
# ============================================================

ranking_results = []

for candidate_id, group in ranking_df.groupby("candidate_id"):

    metrics = evaluate_candidate_ranking(group)

    ranking_results.append({
        "candidate_id": candidate_id,
        "precision_at_5": metrics["precision_at_5"],
        "recall_at_5": metrics["recall_at_5"],
        "ndcg_at_5": metrics["ndcg_at_5"],
        "reciprocal_rank": metrics["reciprocal_rank"]
    })


ranking_metrics_by_candidate = pd.DataFrame(ranking_results)


# ============================================================
# Overall Recommendation Ranking Metrics
# ============================================================

mean_precision_at_5 = ranking_metrics_by_candidate["precision_at_5"].mean()
mean_recall_at_5 = ranking_metrics_by_candidate["recall_at_5"].mean()
mean_ndcg_at_5 = ranking_metrics_by_candidate["ndcg_at_5"].mean()
mrr = ranking_metrics_by_candidate["reciprocal_rank"].mean()


print("RECOMMENDATION RANKING EVALUATION")
print("-" * 45)

print(f"Mean Precision@5: {mean_precision_at_5:.4f}")
print(f"Mean Recall@5   : {mean_recall_at_5:.4f}")
print(f"Mean NDCG@5     : {mean_ndcg_at_5:.4f}")
print(f"MRR             : {mrr:.4f}")

print(
    "\nCandidates evaluated:",
    ranking_metrics_by_candidate["candidate_id"].nunique()
)

RECOMMENDATION RANKING EVALUATION
---------------------------------------------
Mean Precision@5: 0.8800
Mean Recall@5   : 0.6915
Mean NDCG@5     : 0.9782
MRR             : 1.0000

Candidates evaluated: 20


In [11]:
# ============================================================
# Cell 6: Skill Gap Evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Ground truth and system prediction
y_true_gap = skill_gap_df["human_gap"].astype(int)
y_pred_gap = skill_gap_df["system_gap"].astype(int)

# Core classification metrics
gap_accuracy = accuracy_score(y_true_gap, y_pred_gap)
gap_precision = precision_score(y_true_gap, y_pred_gap, zero_division=0)
gap_recall = recall_score(y_true_gap, y_pred_gap, zero_division=0)
gap_f1 = f1_score(y_true_gap, y_pred_gap, zero_division=0)

tn_gap, fp_gap, fn_gap, tp_gap = confusion_matrix(
    y_true_gap,
    y_pred_gap
).ravel()

# Human severity and expert-rating summaries
avg_gap_severity = skill_gap_df["human_gap_severity"].mean()
avg_expert_rating = skill_gap_df["expert_rating"].mean()

# Expert agreement percentage:
# ratings 4 or 5 are treated as positive expert agreement
expert_agreement_pct = (
    (skill_gap_df["expert_rating"] >= 4).mean() * 100
)

print("SKILL GAP EVALUATION")
print("-" * 40)

print(f"Accuracy : {gap_accuracy:.4f}")
print(f"Precision: {gap_precision:.4f}")
print(f"Recall   : {gap_recall:.4f}")
print(f"F1 Score : {gap_f1:.4f}")

print("\nConfusion Matrix Counts")
print(f"TP: {tp_gap}")
print(f"FP: {fp_gap}")
print(f"FN: {fn_gap}")
print(f"TN: {tn_gap}")

print("\nHuman Review Summary")
print(f"Average Human Gap Severity : {avg_gap_severity:.2f}")
print(f"Average Expert Rating      : {avg_expert_rating:.2f}/5")
print(f"Expert Agreement (4 or 5)  : {expert_agreement_pct:.2f}%")

SKILL GAP EVALUATION
----------------------------------------
Accuracy : 0.8350
Precision: 0.6491
Recall   : 0.7400
F1 Score : 0.6916

Confusion Matrix Counts
TP: 37
FP: 20
FN: 13
TN: 130

Human Review Summary
Average Human Gap Severity : 0.98
Average Expert Rating      : 4.84/5
Expert Agreement (4 or 5)  : 96.00%


In [13]:
# ============================================================
# Cell 7: RAG Human Evaluation
# ============================================================

# Rating columns
rag_rating_columns = [
    "relevance_rating",
    "correctness_rating",
    "groundedness_rating",
    "citation_correctness_rating",
    "completeness_rating",
    "clarity_rating",
    "usefulness_rating"
]

# Convert ratings to numeric just to ensure clean calculation
for col in rag_rating_columns:
    rag_df[col] = pd.to_numeric(rag_df[col], errors="coerce")

rag_df["unsupported_claim_count"] = pd.to_numeric(
    rag_df["unsupported_claim_count"],
    errors="coerce"
)

# Mean scores
rag_mean_scores = rag_df[rag_rating_columns].mean()

# Overall mean across all seven dimensions
rag_overall_mean = rag_df[rag_rating_columns].mean(axis=1).mean()

# Percentage of ratings considered good/excellent (4 or 5)
rag_positive_rating_pct = (
    rag_df[rag_rating_columns]
    .ge(4)
    .stack()
    .mean()
    * 100
)

# Unsupported claims
total_unsupported_claims = rag_df["unsupported_claim_count"].sum()

rows_with_unsupported_claims = (
    rag_df["unsupported_claim_count"] > 0
).sum()

print("RAG HUMAN EVALUATION")
print("-" * 40)

for col in rag_rating_columns:
    print(
        f"{col.replace('_', ' ').title():30s}: "
        f"{rag_mean_scores[col]:.2f}/5"
    )

print("\nOverall RAG Mean Rating :", f"{rag_overall_mean:.2f}/5")
print(
    "Positive Ratings (4 or 5):",
    f"{rag_positive_rating_pct:.2f}%"
)

print("\nUnsupported Claim Analysis")
print("Total Unsupported Claims       :", int(total_unsupported_claims))
print(
    "Rows With Unsupported Claims   :",
    f"{rows_with_unsupported_claims}/{len(rag_df)}"
)

RAG HUMAN EVALUATION
----------------------------------------
Relevance Rating              : 5.00/5
Correctness Rating            : 5.00/5
Groundedness Rating           : 5.00/5
Citation Correctness Rating   : 5.00/5
Completeness Rating           : 5.00/5
Clarity Rating                : 5.00/5
Usefulness Rating             : 5.00/5

Overall RAG Mean Rating : 5.00/5
Positive Ratings (4 or 5): 100.00%

Unsupported Claim Analysis
Total Unsupported Claims       : 0
Rows With Unsupported Claims   : 0/20


In [15]:
# ============================================================
# Cell 8: Final Human Evaluation Summary
# ============================================================

final_evaluation_summary = pd.DataFrame([
    {
        "evaluation_component": "Skill Extraction",
        "metric": "Accuracy",
        "value": skill_accuracy
    },
    {
        "evaluation_component": "Skill Extraction",
        "metric": "Precision",
        "value": skill_precision
    },
    {
        "evaluation_component": "Skill Extraction",
        "metric": "Recall",
        "value": skill_recall
    },
    {
        "evaluation_component": "Skill Extraction",
        "metric": "F1 Score",
        "value": skill_f1
    },

    {
        "evaluation_component": "Recommendation Ranking",
        "metric": "Precision@5",
        "value": mean_precision_at_5
    },
    {
        "evaluation_component": "Recommendation Ranking",
        "metric": "Recall@5",
        "value": mean_recall_at_5
    },
    {
        "evaluation_component": "Recommendation Ranking",
        "metric": "NDCG@5",
        "value": mean_ndcg_at_5
    },
    {
        "evaluation_component": "Recommendation Ranking",
        "metric": "MRR",
        "value": mrr
    },

    {
        "evaluation_component": "Skill Gap",
        "metric": "Accuracy",
        "value": gap_accuracy
    },
    {
        "evaluation_component": "Skill Gap",
        "metric": "Precision",
        "value": gap_precision
    },
    {
        "evaluation_component": "Skill Gap",
        "metric": "Recall",
        "value": gap_recall
    },
    {
        "evaluation_component": "Skill Gap",
        "metric": "F1 Score",
        "value": gap_f1
    },
    {
        "evaluation_component": "Skill Gap",
        "metric": "Average Expert Rating",
        "value": avg_expert_rating
    },

    {
        "evaluation_component": "RAG",
        "metric": "Overall Mean Rating",
        "value": rag_overall_mean
    },
    {
        "evaluation_component": "RAG",
        "metric": "Positive Rating Percentage",
        "value": rag_positive_rating_pct / 100
    },
    {
        "evaluation_component": "RAG",
        "metric": "Unsupported Claims",
        "value": total_unsupported_claims
    }
])

print("FINAL HUMAN EVALUATION SUMMARY")
print("=" * 65)

display(final_evaluation_summary)

FINAL HUMAN EVALUATION SUMMARY


,evaluation_component,metric,value
0,Skill Extraction,Accuracy,0.835000
1,Skill Extraction,Precision,0.881119
2,Skill Extraction,Recall,0.887324
3,Skill Extraction,F1 Score,0.884211
4,Recommendation Ranking,Precision@5,0.880000
5,Recommendation Ranking,Recall@5,0.691486
6,Recommendation Ranking,NDCG@5,0.978206
7,Recommendation Ranking,MRR,1.000000
8,Skill Gap,Accuracy,0.835000
9,Skill Gap,Precision,0.649123


In [17]:
# ============================================================
# Cell 9: Export Final Human Evaluation Results
# ============================================================

# ------------------------------------------------------------
# 1. Overall human-evaluation summary
# ------------------------------------------------------------
summary_file = FINAL_EVAL_DIR / "final_human_evaluation_summary.csv"

final_evaluation_summary.to_csv(
    summary_file,
    index=False
)


# ------------------------------------------------------------
# 2. Candidate-level recommendation ranking metrics
# ------------------------------------------------------------
ranking_metrics_file = (
    FINAL_EVAL_DIR /
    "final_recommendation_ranking_metrics_by_candidate.csv"
)

ranking_metrics_by_candidate.to_csv(
    ranking_metrics_file,
    index=False
)


# ------------------------------------------------------------
# 3. RAG dimension-level summary
# ------------------------------------------------------------
rag_dimension_summary = pd.DataFrame({
    "rag_dimension": rag_rating_columns,
    "mean_rating": [
        rag_mean_scores[col]
        for col in rag_rating_columns
    ]
})

rag_dimension_file = (
    FINAL_EVAL_DIR /
    "final_rag_human_evaluation_summary.csv"
)

rag_dimension_summary.to_csv(
    rag_dimension_file,
    index=False
)


# ------------------------------------------------------------
# 4. Validation / audit summary
# ------------------------------------------------------------
evaluation_validation = pd.DataFrame([
    {
        "evaluation_component": "Skill Extraction",
        "reviewed_rows": len(skill_extraction_df),
        "human_labels_complete": skill_extraction_df[
            "human_skill_present"
        ].notna().all()
    },
    {
        "evaluation_component": "Recommendation Ranking",
        "reviewed_rows": len(ranking_df),
        "human_labels_complete": (
            ranking_df["human_relevant"].notna().all()
            and
            ranking_df["human_relevance_grade"].notna().all()
        )
    },
    {
        "evaluation_component": "Skill Gap",
        "reviewed_rows": len(skill_gap_df),
        "human_labels_complete": (
            skill_gap_df["human_gap"].notna().all()
            and
            skill_gap_df["human_gap_severity"].notna().all()
            and
            skill_gap_df["expert_rating"].notna().all()
        )
    },
    {
        "evaluation_component": "RAG",
        "reviewed_rows": len(rag_df),
        "human_labels_complete": (
            rag_df[rag_rating_columns].notna().all().all()
            and
            rag_df["unsupported_claim_count"].notna().all()
        )
    }
])

validation_file = (
    FINAL_EVAL_DIR /
    "final_human_evaluation_validation.csv"
)

evaluation_validation.to_csv(
    validation_file,
    index=False
)


# ------------------------------------------------------------
# Print export confirmation
# ------------------------------------------------------------
print("FINAL HUMAN EVALUATION EXPORTS")
print("=" * 60)

print("1.", summary_file.name)
print("2.", ranking_metrics_file.name)
print("3.", rag_dimension_file.name)
print("4.", validation_file.name)

print("\nSaved to:")
print(FINAL_EVAL_DIR)

print("\nVALIDATION")
display(evaluation_validation)

FINAL HUMAN EVALUATION EXPORTS
1. final_human_evaluation_summary.csv
2. final_recommendation_ranking_metrics_by_candidate.csv
3. final_rag_human_evaluation_summary.csv
4. final_human_evaluation_validation.csv

Saved to:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Human_Evaluation

VALIDATION


,evaluation_component,reviewed_rows,human_labels_complete
0,Skill Extraction,200,True
1,Recommendation Ranking,300,True
2,Skill Gap,200,True
3,RAG,20,True


In [19]:
# ============================================================
# Cell 10: Final Export Read-Back Validation
# ============================================================

exported_files = {
    "Human Evaluation Summary":
        FINAL_EVAL_DIR / "final_human_evaluation_summary.csv",

    "Ranking Metrics":
        FINAL_EVAL_DIR / "final_recommendation_ranking_metrics_by_candidate.csv",

    "RAG Evaluation Summary":
        FINAL_EVAL_DIR / "final_rag_human_evaluation_summary.csv",

    "Human Evaluation Validation":
        FINAL_EVAL_DIR / "final_human_evaluation_validation.csv"
}

print("FINAL EXPORT READ-BACK VALIDATION")
print("=" * 65)

all_exports_valid = True

for name, file_path in exported_files.items():

    exists = file_path.exists()

    if exists:
        check_df = pd.read_csv(file_path)
        rows = len(check_df)
        columns = len(check_df.columns)
        readable = rows > 0 and columns > 0
    else:
        rows = 0
        columns = 0
        readable = False

    valid = exists and readable

    if not valid:
        all_exports_valid = False

    print(
        f"{name:30s} | "
        f"Exists: {str(exists):5s} | "
        f"Shape: ({rows}, {columns}) | "
        f"Valid: {valid}"
    )

print("\n" + "=" * 65)

if all_exports_valid:
    print("FINAL STATUS: ALL HUMAN EVALUATION EXPORTS VALID")
else:
    print("FINAL STATUS: CHECK FAILED EXPORTS")

FINAL EXPORT READ-BACK VALIDATION
Human Evaluation Summary       | Exists: True  | Shape: (16, 3) | Valid: True
Ranking Metrics                | Exists: True  | Shape: (20, 5) | Valid: True
RAG Evaluation Summary         | Exists: True  | Shape: (7, 2) | Valid: True
Human Evaluation Validation    | Exists: True  | Shape: (4, 3) | Valid: True

FINAL STATUS: ALL HUMAN EVALUATION EXPORTS VALID
